In [ ]:
import pandas as pd
from scipy.stats import pearsonr

def strict_major_vote(lst):
    if isinstance(lst, list):
        counts = pd.Series(lst).value_counts()
        max_count = counts.max()
        candidates = counts[counts == max_count].index.tolist()
        if len(candidates) > 1:
            return int(sum(candidates) / len(candidates))
        return candidates[0]
    return lst

def process_dataframe(df, col1, col2):
    # Step 1: Convert lists of ints to single int using strict major vote

    df[col2] = df[col2].apply(strict_major_vote)

    # Step 2: Calculate Pearson correlation
    correlation, _ = pearsonr(df[col1], df[col2])

    # Step 3: Report the correlation
    print(f"Pearson correlation between {col1} and {col2}: {correlation}")
    return correlation

In [ ]:
import os
import pandas as pd
import seaborn as sns

import matplotlib.pyplot as plt

def load_and_process_data(dataset_name, ground_truth_names, proposed_names):
    base_dir = f'data/{dataset_name}'
    result_dir = f'results/scoring/{dataset_name}/comparisons'
    
    if not os.path.exists(result_dir):
        os.makedirs(result_dir)
    
    for api_name in os.listdir(base_dir):
        api_dir = os.path.join(base_dir, api_name)
        if os.path.isdir(api_dir):
            for csv_file in os.listdir(api_dir):
                if csv_file.endswith('.csv'):
                    file_path = os.path.join(api_dir, csv_file)
                    df = pd.read_csv(file_path)
                    
                    correlation_matrix = pd.DataFrame(index=ground_truth_names, columns=[x.split("_score")[0] for x in proposed_names])
                    
                    for gt_name in ground_truth_names:
                        for prop_name in proposed_names:
                            prop_name_base = prop_name.split("_score")[0]
                            correlation, _ = process_dataframe(df, gt_name, prop_name)
                            correlation_matrix.loc[gt_name, prop_name_base] = correlation
                    
                    matrix_name = os.path.splitext(csv_file)[0]
                    save_path = os.path.join(result_dir, api_name, matrix_name)
                    
                    if not os.path.exists(os.path.dirname(save_path)):
                        os.makedirs(os.path.dirname(save_path))
                    
                    plt.figure(figsize=(10, 8))
                    sns.heatmap(correlation_matrix.astype(float), annot=True, cmap='coolwarm')
                    plt.title(f'Correlation Matrix for {matrix_name}')
                    plt.savefig(f'{save_path}.png')
                    plt.close()